In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

In [ ]:
load_dotenv()
api = os.getenv("APIKEY")
Ch_Handler = '@AJpluskibreet'
#Ch_Handler = input()

In [ ]:
def get_videos_ids(playlist_id):
    base_url =f"https://youtube.googleapis.com/youtube/v3/playlistItems?part=contentDetails&maxResults=50&playlistId={playlist_id}&key={api}"
    pageToken = None
    video_ids = []
    try:
        while True:
          url = base_url
          if pageToken:
              url += f"&pageToken={pageToken}"

          response = requests.get(url)
          response.raise_for_status()  # Check if the request was successful
          data = response.json()
          for item in data.get("items", []):
              video_ids.append(item["contentDetails"]["videoId"])

          pageToken = data.get("nextPageToken")

          if not pageToken:
              break
        return video_ids    

    except requests.exceptions.RequestException as e:
        raise Exception(f"An error occurred: {e}")    
   

if __name__ == "__main__":
    playlist_id = get_playlist_id()
    print(get_videos_ids(playlist_id))


print(len(get_videos_ids(playlist_id)))

NameError: name 'get_playlist_id' is not defined

In [ ]:
def batch_list(video_ids_lst, batch_size):
    for video_id in range(0, len(video_ids_lst), batch_size):
        yield video_ids_lst[video_id:video_id + batch_size]


def extract_video_details(video_ids):
    video_details = []
    for batch in batch_list(video_ids, 50):
        ids = ",".join(batch)
        url = f"https://youtube.googleapis.com/youtube/v3/videos?part=snippet,contentDetails,statistics&id={ids}&key="
        try:
            response = requests.get(url)
            response.raise_for_status()  # Check if the request was successful
            data = response.json()
            for item in data.get("items", []):
                video_details.append({
                    "videoId": item["id"],
                    "title": item["snippet"]["title"],
                    "publishedAt": item["snippet"]["publishedAt"],
                    "duration": item["contentDetails"]["duration"],
                    "viewCount": item["statistics"].get("viewCount", 0),
                    "likeCount": item["statistics"].get("likeCount", 0),
                    "commentCount": item["statistics"].get("commentCount", 0)
                })
        except requests.exceptions.RequestException as e:
            raise Exception(f"An error occurred: {e}")
    return video_details

if __name__ == "__main__":
    playlist_id = get_playlist_id()
    video_ids = get_videos_ids(playlist_id)
    details = extract_video_details(video_ids)
    print(details)